In [1]:
import pandas as pd
import requests
import json
import os
import boto3
import re
from cities import cities
from dotenv import load_dotenv
from sqlalchemy import create_engine
import numpy as np

In [2]:
load_dotenv()
load_dotenv(dotenv_path=os.path.expanduser("~/.secret_access/kayak_aws"))


True

# Weather data

In [3]:
# Get the latitude and longitude of the cities
cities_location = []

headers = {
    "User-Agent": "QHA"
}

for city in cities:
    r_location = requests.get(f"https://nominatim.openstreetmap.org/search?format=json&city={city}", headers=headers)
    location_data = r_location.json()

    current_city = {
        "name": location_data[0]['name'],
        "lat": location_data[0]['lat'],
        "lon": location_data[0]['lon'],
    }

    cities_location.append(current_city)
print(cities_location)



[{'name': 'Mont-Saint-Michel', 'lat': '46.7798558', 'lon': '-75.3362610'}, {'name': 'St. Malo', 'lat': '49.3146950', 'lon': '-96.9538228'}, {'name': 'Bayeux', 'lat': '49.2764624', 'lon': '-0.7024738'}, {'name': 'Le Havre', 'lat': '49.4938975', 'lon': '0.1079732'}, {'name': 'Rouen', 'lat': '49.4404591', 'lon': '1.0939658'}, {'name': 'Paris', 'lat': '48.8588897', 'lon': '2.3200410'}, {'name': 'Amiens', 'lat': '49.8941708', 'lon': '2.2956951'}, {'name': 'Lille', 'lat': '50.6365654', 'lon': '3.0635282'}, {'name': 'Strasbourg', 'lat': '48.5846140', 'lon': '7.7507127'}, {'name': 'Château du Haut-Kœnigsbourg', 'lat': '48.2495226', 'lon': '7.3454923'}, {'name': 'Colmar', 'lat': '48.0777517', 'lon': '7.3579641'}, {'name': 'Eguisheim', 'lat': '48.0447968', 'lon': '7.3079618'}, {'name': 'Besançon', 'lat': '47.2380222', 'lon': '6.0243622'}, {'name': 'Dijon', 'lat': '47.3215806', 'lon': '5.0414701'}, {'name': 'Annecy', 'lat': '45.8992348', 'lon': '6.1288847'}, {'name': 'Grenoble', 'lat': '45.187560

In [4]:
df_location = pd.DataFrame(cities_location)
df_location

,name,lat,lon
0,Mont-Saint-Michel,46.7798558,-75.3362610
1,St. Malo,49.3146950,-96.9538228
2,Bayeux,49.2764624,-0.7024738
3,Le Havre,49.4938975,0.1079732
4,Rouen,49.4404591,1.0939658
5,Paris,48.8588897,2.3200410
6,Amiens,49.8941708,2.2956951
7,Lille,50.6365654,3.0635282
8,Strasbourg,48.5846140,7.7507127
9,Château du Haut-Kœnigsbourg,48.2495226,7.3454923


In [19]:
api_weather_key = os.getenv("open_weather_api_key")

params = {
    "appid": api_weather_key,
    "units": "metric",
    "lang": "fr",
    "cnt": 5,
}

lat = 44.8333
lon = -0.5667
cnt = 16

url_test = f"https://api.openweathermap.org/data/2.5/forecast?lat=44.8333&lon=-0.5667&units=metric&lang=fr&cnt=7&appid={api_weather_key}"

r = requests.get(url_test)
data = r.json()

data['list']



KeyError: 'list'

In [6]:
all_weather = []
weather_params = {
    "appid": api_weather_key,
    "units": "metric",
    "lang": "fr",
    "cnt": 7,
}

for index,weather in enumerate(cities_location):
    temp_score = 0
    rain = 0
    wind = 0
    wind_score = 0
    rain_score = 0
    
    r_weather = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={weather['lat']}&lon={weather['lon']}", params=weather_params)
    current_weather = r_weather.json()['list']

    for i in range(len(current_weather)):
        temp_score = round(temp_score + (current_weather[i]['main']['feels_like'] / len(current_weather)), 2)
        rain = current_weather[i].get('rain', 0)
        
        if rain != 0:
            rain_key = list(current_weather[i]['rain'].keys())[0]
            match = re.search(r'\d+', rain_key)
            hours = 0

            if match:
                hours = int(match.group())

            rain_daily_score = hours * current_weather[i]['rain'][rain_key]
            rain_score = round(rain_score + (rain_daily_score / len(current_weather)), 2)
        else:
            rain_score = rain_score + 0

        wind = round(wind + (current_weather[i]['wind']['speed'] / len(current_weather)), 2)

        beaufort_score = 0 # Beaufort scale
        if current_weather[i]['wind']['speed'] == 0:
            beaufort_score = 0
        elif current_weather[i]['wind']['speed'] < 5:
            beaufort_score = 1
        elif current_weather[i]['wind']['speed'] < 11:
            beaufort_score = 2
        elif current_weather[i]['wind']['speed'] < 19:
            beaufort_score = 3
        elif current_weather[i]['wind']['speed'] < 28:
            beaufort_score = 4
        elif current_weather[i]['wind']['speed'] < 38:
            beaufort_score = 5
        elif current_weather[i]['wind']['speed'] < 49:
            beaufort_score = 6
        elif current_weather[i]['wind']['speed'] < 61:
            beaufort_score = 7
        elif current_weather[i]['wind']['speed'] < 74:
            beaufort_score = 8
        elif current_weather[i]['wind']['speed'] < 88:
            beaufort_score = 9
        elif current_weather[i]['wind']['speed'] < 102:
            beaufort_score = 10
        elif current_weather[i]['wind']['speed'] < 117:
            beaufort_score = 11
        elif current_weather[i]['wind']['speed'] > 117:
            beaufort_score = 12

        if current_weather[i]['main']['feels_like'] < 25 or current_weather[i]['wind']['speed'] > 30:
            wind_penalty = beaufort_score
        else:
            wind_penalty = 0

        wind_score = round(wind_score + (wind_penalty / len(current_weather)), 2)

    weather_score = round(temp_score - rain_score - wind_score, 3)


    current_weather_data = {
        "index" : index,
        "name" : weather['name'],
        "temperature_mean" : temp_score,
        "rain_mean" : rain_score,
        "wind_score" : wind_score,
        "score" : weather_score,
    }
    all_weather.append(current_weather_data)
       
all_weather


[{'index': 0,
  'name': 'Mont-Saint-Michel',
  'temperature_mean': 7.56,
  'rain_mean': 0.83,
  'wind_score': 0.98,
  'score': 5.75},
 {'index': 1,
  'name': 'St. Malo',
  'temperature_mean': 10.57,
  'rain_mean': 0.58,
  'wind_score': 1.43,
  'score': 8.56},
 {'index': 2,
  'name': 'Bayeux',
  'temperature_mean': 10.83,
  'rain_mean': 5.82,
  'wind_score': 0.98,
  'score': 4.03},
 {'index': 3,
  'name': 'Le Havre',
  'temperature_mean': 11.32,
  'rain_mean': 4.73,
  'wind_score': 0.98,
  'score': 5.61},
 {'index': 4,
  'name': 'Rouen',
  'temperature_mean': 12.09,
  'rain_mean': 4.66,
  'wind_score': 0.98,
  'score': 6.45},
 {'index': 5,
  'name': 'Paris',
  'temperature_mean': 11.48,
  'rain_mean': 4.64,
  'wind_score': 0.98,
  'score': 5.86},
 {'index': 6,
  'name': 'Amiens',
  'temperature_mean': 10.51,
  'rain_mean': 0.55,
  'wind_score': 0.98,
  'score': 8.98},
 {'index': 7,
  'name': 'Lille',
  'temperature_mean': 11.47,
  'rain_mean': 0.82,
  'wind_score': 1.13,
  'score': 9.52

In [7]:
df_weather = pd.DataFrame(all_weather)
df_weather = df_weather.sort_values(by='score', ascending=False)
df_weather

,index,name,temperature_mean,rain_mean,wind_score,score
22,22,Avignon,20.21,0.16,1.29,18.76
24,24,Nîmes,19.88,0.00,1.14,18.74
23,23,Uzès,19.28,0.19,1.28,17.81
20,20,Marseille,20.11,0.39,2.03,17.69
27,27,Collioure,19.03,0.37,0.98,17.68
21,21,Aix-en-Provence,19.19,0.14,1.43,17.62
25,25,Aigues-Mortes,19.75,0.31,2.03,17.41
18,18,Bormes-les-Mimosas,18.87,0.00,1.88,16.99
19,19,Cassis,19.22,0.49,2.03,16.70
26,26,Saintes-Maries-de-la-Mer,18.66,0.35,2.03,16.28


In [8]:
df_weather.to_csv("export/weather.csv", index=False)

# Hotel data

In [9]:
!python hotel.py

2025-05-21 09:17:46 [scrapy.utils.log] INFO: Scrapy 2.11.1 started (bot: scrapybot)
2025-05-21 09:17:46 [scrapy.utils.log] INFO: Versions: lxml 5.2.1.0, libxml2 2.13.1, cssselect 1.2.0, parsel 1.8.1, w3lib 2.1.2, Twisted 23.10.0, Python 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:28:27) [Clang 14.0.6 ], pyOpenSSL 24.2.1 (OpenSSL 3.0.15 3 Sep 2024), cryptography 43.0.0, Platform macOS-10.16-x86_64-i386-64bit
2025-05-21 09:17:46 [scrapy.addons] INFO: Enabled addons:
[]
2025-05-21 09:17:46 [py.warnings] WARNING: /opt/anaconda3/lib/python3.12/site-packages/scrapy/utils/request.py:254: ScrapyDeprecationWarning: '2.6' is a deprecated value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting.

It is also the default value. In other words, it is normal to get this warning if you have not defined a value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting. This is so for backward compatibility reasons, but it will change in a future version of Scrapy.

See the documentati

# Merge data in one csv

In [10]:
with open("export/hotels.json", "r") as f:
    hotel_data = json.load(f)

df_hotel = pd.DataFrame(hotel_data)
df_hotel = df_hotel.rename(columns={""
    "name": "hotel_name",
    "link": "hotel_link",
    "latitude": "lat",
    "longitude": "lon",
    "description": "desc",
})
df_hotel[df_hotel["city_id"]== 0]

,city_id,city,hotel_name,hotel_link,lat,lon,desc,stars,rating
309,0,Mont Saint Michel,Maison au pied du Mont Saint Michel,https://www.booking.com/hotel/fr/maison-au-pie...,48.612673580235,-1.485530645404,L’hébergement Maison au pied du Mont Saint Mic...,0,"9,3"
312,0,Mont Saint Michel,Auberge de la Baie,https://www.booking.com/hotel/fr/auberge-de-la...,48.61599317655586,-1.4882531762123108,"Situé dans la campagne normande, cet hôtel dis...",2,"8,3"
313,0,Mont Saint Michel,Mon Saint Michel,https://www.booking.com/hotel/fr/mon-saint-mic...,48.613627,-1.485791,"Offrant une vue sur le jardin, l’établissement...",0,"8,8"
314,0,Mont Saint Michel,Maison au pied du Mont Saint Michel 2,https://www.booking.com/hotel/fr/maison-au-pie...,48.6156117,-1.4885121,"Offrant une vue sur le jardin, l’hébergement M...",0,"9,3"
315,0,Mont Saint Michel,Roulotte au pied du Mont Saint Michel,https://www.booking.com/hotel/fr/roulotte-au-p...,48.613969646033,-1.486292458336,L’hébergement Roulotte au pied du Mont Saint M...,0,"9,5"
316,0,Mont Saint Michel,gites 2 beauvoir,https://www.booking.com/hotel/fr/gites-2-beauv...,48.5976913,-1.5047171,Hébergement géré par un particulier,0,"8,7"
317,0,Mont Saint Michel,Apparthôtel Mont Saint Michel - Résidence Fleu...,https://www.booking.com/hotel/fr/residence-fle...,48.596482169806976,-1.5031469166023044,L’Apparthôtel Mont Saint Michel - Résidence Fl...,3,"8,3"
318,0,Mont Saint Michel,Le Marquis De La Guintre,https://www.booking.com/hotel/fr/le-marquis-de...,48.6248647057706,-1.44534185528755,"Situé à Courtils, l’établissement Le Marquis D...",0,"8,8"
319,0,Mont Saint Michel,Vent des Grèves,https://www.booking.com/hotel/fr/vent-des-grev...,48.615403,-1.49144,"Offrant une vue sur le jardin, l’établissement...",0,"9,1"
320,0,Mont Saint Michel,A l ombre du Mont Saint Michel,https://www.booking.com/hotel/fr/a-l-ombre-du-...,48.6156258,-1.4651207,"Situé à Huisnes-sur-Mer, l’établissement A l o...",0,"9,5"


In [11]:
df_weather = df_weather.rename(columns={
    "index": "city_id",
    "name": "city_name",
})
df_weather[df_weather["city_id"]== 0]


,city_id,city_name,temperature_mean,rain_mean,wind_score,score
0,0,Mont-Saint-Michel,7.56,0.83,0.98,5.75


In [12]:
df_global = pd.merge(df_weather, df_hotel, on='city_id', how='inner')
df_global = df_global.drop(columns=["city"])
df_global = df_global.rename(columns={"city_name": "city"})
df_global["stars"] = df_global["stars"].replace(0, np.nan)
df_global["rating"] = df_global["rating"].astype(str).str.replace(",", ".")
df_global["rating"] = pd.to_numeric(df_global["rating"], errors="coerce")
df_global.reset_index(drop=False, inplace=True)
df_global.rename(columns={"index": "Id"}, inplace=True)

df_global.sample(5)


,Id,city_id,city,temperature_mean,rain_mean,wind_score,score,hotel_name,hotel_link,lat,lon,desc,stars,rating
128,128,21,Aix-en-Provence,19.19,0.14,1.43,17.62,HOTEL RESTAURANT OLYMPE,https://www.booking.com/hotel/fr/restaurant-ol...,43.508036,5.439455,L’établissement HOTEL RESTAURANT OLYMPE se tro...,2.0,7.7
332,332,29,Tarascon-sur-Ariège,16.54,1.92,0.98,13.64,la maison d'Anna chambres d hôtes,https://www.booking.com/hotel/fr/la-maison-d-3...,42.9509120273479,1.57907545566559,Hébergement géré par un particulier,NaN,9.3
135,135,21,Aix-en-Provence,19.19,0.14,1.43,17.62,Le Mini - Un voyage en Provence,https://www.booking.com/hotel/fr/studio-terras...,43.528465619432,5.452499735058,Hébergement géré par un particulier,NaN,8.8
452,452,17,Rougon,13.30,0.20,0.98,12.12,Château de Trigance,https://www.booking.com/hotel/fr/chateau-de-tr...,43.76192795006577,6.444082260131836,Le château de Trigance est situé dans le villa...,3.0,9.4
556,556,12,Besançon,15.24,4.05,0.98,10.21,L'Amour d'Or Centre Historique,https://www.booking.com/hotel/fr/amour-d-or-ce...,47.23364697556,6.029169180826,Hébergement géré par un particulier,NaN,9.7


In [13]:
df_global.to_csv("export/weather_and_hotels.csv", index=False)

# Data to S3

In [14]:
session = boto3.Session()

In [15]:
s3 = session.resource("s3")

In [16]:
s3.Bucket("qha-kayak").upload_file("export/weather_and_hotels.csv", "weather_and_hotels.csv")

# Send to database

In [17]:
USERNAME = os.getenv("aws_kayak_user_name")
PASSWORD = os.getenv("aws_kayak_password")
HOSTNAME = os.getenv("aws_kayak_host_name")
DB_NAME = os.getenv("aws_kayak_db_name")
PORT = 5432

engine = create_engine(f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOSTNAME}:{PORT}/{DB_NAME}")

df_global.to_sql("weather_and_hotels", engine, if_exists="replace", index=False)


875

# Import data of my database

In [18]:


#engine = create_engine(f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOSTNAME}:{PORT}/{DB_NAME}")
df = pd.read_sql("SELECT * FROM weather_and_hotels", engine)
df
#print(df)


,Id,city_id,city,temperature_mean,rain_mean,wind_score,score,hotel_name,hotel_link,lat,lon,desc,stars,rating
0,0,22,Avignon,20.21,0.16,1.29,18.76,Le Saint Roch 1 - AC CLIM - WIFI - 50m Centre ...,https://www.booking.com/hotel/fr/appartement-l...,43.942291,4.8084435,L’hébergement Le Saint Roch 1 - AC CLIM - WIFI...,NaN,8.6
1,1,22,Avignon,20.21,0.16,1.29,18.76,Charmant studio 25 m2 centre ville Avignon,https://www.booking.com/hotel/fr/charmant-stud...,43.9450644,4.8138442,L’hébergement Charmant studio 25 m2 centre vil...,NaN,8.2
2,2,22,Avignon,20.21,0.16,1.29,18.76,Mercure Avignon Gare TGV,https://www.booking.com/hotel/fr/expresshiavig...,43.92929,4.784118000000035,Le Mercure Avignon Gare TGV vous accueille à A...,4.0,8.6
3,3,22,Avignon,20.21,0.16,1.29,18.76,Hotel de l'île,https://www.booking.com/hotel/fr/de-l-ile-avig...,43.9569418,4.802945099999988,L’établissement Hotel de l'île vous accueille ...,2.0,8.1
4,4,22,Avignon,20.21,0.16,1.29,18.76,Appartement Lumineux Avignon Intra-muros au calme,https://www.booking.com/hotel/fr/au-calme-avec...,43.94411276236114,4.812503901835048,Hébergement géré par un particulier,NaN,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
870,870,2,Bayeux,10.83,5.82,0.98,4.03,Gites les Pourquoi Pas - Résidence de Tourisme...,https://www.booking.com/hotel/fr/gites-les-pou...,49.2815226,-0.7081344,Proposant une connexion Wi-Fi gratuite et des ...,NaN,9.1
871,871,2,Bayeux,10.83,5.82,0.98,4.03,Tiny house au cœur de Bayeux,https://www.booking.com/hotel/fr/tiny-house-au...,49.2755584,-0.6992038,Hébergement géré par un particulier,NaN,9.7
872,872,2,Bayeux,10.83,5.82,0.98,4.03,Domaine de Bayeux,https://www.booking.com/hotel/fr/domaine-de-ba...,49.27232559999999,-0.6985101000000213,Le Domaine de Bayeux occupe une maison du XVII...,3.0,9.2
873,873,2,Bayeux,10.83,5.82,0.98,4.03,Villa Des Ursulines,https://www.booking.com/hotel/fr/appartement-d...,49.27646774792992,-0.7055658153442437,Hébergement géré par un particulier,NaN,8.3
